<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/01-glowtts-from-scratch/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import soundfile as sf

from TTS.config import load_config
from TTS.tts.models.glow_tts import GlowTTS
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

In [ ]:
CONFIG_PATH = "config.json"
MODEL_PATH = "best_model.pth"
OUT_PATH = "output.wav"

In [ ]:
config = load_config(CONFIG_PATH)

model = GlowTTS(config)
ckpt = torch.load(MODEL_PATH, map_location="cpu")

if "model" in ckpt:
    model.load_state_dict(ckpt["model"])
else:
    model.load_state_dict(ckpt)

model.eval()

In [ ]:
tokenizer = TTSTokenizer(config)

In [ ]:
text = "have you played that new game called animal well which has no combat but amazing visuals"

tokens = tokenizer.text_to_ids(text)
tokens = torch.LongTensor(tokens).unsqueeze(0)

In [ ]:
with torch.no_grad():
  out = model.inference(tokens)

mel = out["mel"]

In [ ]:
ap = AudioProcessor(**config.audio)
wav = ap.inv_melspectrogram(mel[0].cpu().numpy())

In [ ]:
sf.write(OUT_PATH, wav, config.audio["sample_rate"])
print("saved:", OUT_PATH)

In [ ]:
from IPython.display import Audio
Audio(OUT_PATH)